In [53]:
import pandas as pd
import os
import re

In [ ]:
def extract_date(text):
    pattern = r'(?<!\d)(\d{4})(0[1-9]|1[0-2])'
    match = re.search(pattern, text)
    if match:
        return match.group(1), match.group(2)
    return None

def check_columns(columns, final_columns):
    if set(columns) == set(final_columns):
        return True
    else:
        return False
    
def add_date_columns(df, filename):
    year, month = extract_date(filename)
    df.insert(0, "source", filename)
    df.insert(1, "year", year)
    df.insert(2, "month", month)
    return df

def not_in_columns(df1, df2):
    col1 = set(df1.columns)
    col2 = set(df2.columns)
    not_in_1 = col1.difference(col2)
    not_in_2 = col2.difference(col1)
    return not_in_1, not_in_2

In [55]:
folder_path = 'csv'

listing_df = None
listing_final_df = None

sold_df = None
sold_final_df= None

sold_filled_df = None
sold_fill_final_df = None


In [ ]:
for filename in os.listdir(folder_path):
    print("Processing file:", filename)
    if filename.endswith('.csv') and filename.startswith('CRMLSListing'):
        if listing_final_df is not None:
            listing_df = pd.read_csv(folder_path + '/' + filename)
            if check_columns(listing_df.columns, listing_final_df.columns):
                listing_df = add_date_columns(listing_df, filename)
                listing_final_df = pd.concat([listing_final_df, listing_df])
                print("Concatenated listing_final_df with file:", filename)
            else:
                print("Listing columns do not match:", filename, len(listing_df.columns))

        else:
            print("Initializing listing_final_df with file:", filename)
            listing_final_df = pd.read_csv(folder_path + '/' + filename)
            listing_final_df = add_date_columns(listing_final_df, filename)
        
    if filename.startswith('CRMLSSold') and filename.endswith('_filled.csv'):
        if sold_fill_final_df is not None:
            sold_filled_df = pd.read_csv(folder_path + '/' + filename)
            if check_columns(sold_filled_df.columns, sold_fill_final_df.columns):
                sold_filled_df = add_date_columns(sold_filled_df, filename)
                sold_fill_final_df = pd.concat([sold_fill_final_df, sold_filled_df])
                print("Concatenated sold_fill_final_df with file:", filename)
            else:
                print("Sold filled columns do not match:", filename, len(sold_filled_df.columns))
        else:
            print("Initializing sold_fill_final_df with file:", filename)
            sold_fill_final_df = pd.read_csv(folder_path + '/' + filename)
            sold_fill_final_df = add_date_columns(sold_fill_final_df, filename)
            print("Sold filled columns do not match:", filename, len(sold_filled_col))
        
    if filename.startswith('CRMLSSold') and not filename.endswith('_filled.csv'):
        if sold_final_df is not None:
            sold_df = pd.read_csv(folder_path + '/' + filename)
            if check_columns(sold_df.columns, sold_final_df.columns):
                sold_df = add_date_columns(sold_df, filename)
                sold_final_df = pd.concat([sold_final_df, sold_df])
                print("Concatenated sold_final_df with file:", filename)
            else:
                print("Sold columns do not match:", filename, len(sold_final_df.columns))
        else:  # initialize sold_final_df and sold_col
            print("Initializing sold_final_df with file:", filename)
            sold_final_df = pd.read_csv(folder_path + '/' + filename)
            sold_final_df = add_date_columns(sold_final_df, filename)

In [ ]:
df1 = pd.read_csv("new_extracts\CRMLSListing202401.csv")
df2 = pd.read_csv("new_extracts\CRMLSListing202402.csv")

not_in_columns(df1,df2)

In [ ]:
listing_final_df.to_csv('20260929_listing_output.csv', index=False)
sold_final_df.to_csv('20260929_sold_output.csv', index=False)
sold_fill_final_df.to_csv('20260929_sold_fill_output.csv', index=False)

In [47]:
sold_fill_final_df['OriginatingSystemSubName'] = ''
sold_fill_final_df['OriginatingSystemName'] = ''
sold_final_df['latfilled'] = ''
sold_final_df['lonfilled'] = ''

In [48]:
complete_sold_df = pd.concat([sold_final_df, sold_fill_final_df], ignore_index=True)

In [ ]:
listing_final_df.to_csv('20260929_listing_complete.csv', index=False)
complete_sold_df.to_csv('20260929_sold_complete.csv', index=False)

In [ ]:
#Residential Only
residential_sold_df = complete_sold_df[complete_sold_df['PropertyType'] == 'Residential']
residential_sold_df.to_csv('20260929_residential_sold_output.csv', index=False)
residential_listing_df = listing_final_df[listing_final_df['PropertyType'] == 'Residential']
residential_listing_df.to_csv('20260929_residential_listing_output.csv', index=False)


Used Only Data from Jan 2024 to March 2026.
| csv                        | row count | action                            |
|----------------------------|-----------|-----------------------------------|
| listing_output             | 764666    | concat months                     |
| sold_output                | 454040    | concat months                     |
| sold_fill_output           | 113751    | concat months                     |
| listing_complete           | 764666    | contains complete listing         |
| sold_complete              | 567791    | contains complete sold and filled |
| residential_sold_output    | 380975    | filter by Residential             |
| residential_listing_output | 485146    | filter by Residential             |